# GAIA Workshop Notebook — SystemWeaver-aligned


Trustworthy AI for requirements traceability, without baking question-specific answers into the code.

### Pipeline
```
question (natural language)
     ↓  plan_question()               # rule-based planner (workshop demo)
QuestionPolicy (root, targets, support, claim_relation_sids from chosen whitelists)
     ↓  extract_candidate_subgraph()  # whitelist-constrained reachability + root-anchored paths
bounded subgraph
     ↓  build_evidence_packet()
evidence packet
     ↓  analyze_with_llm()            # strict JSON schema (OpenAI Responses / LM Studio)
parsed answer
     ↓  validate_llm_output()         # every cited id and edge must exist in the packet
review decision
```

### What is pre-coded vs data-driven

**Pre-coded workshop demo**
- `plan_question()` is a deterministic, rule-based planner. It uses the phrase list in `data/metamodel.json → type_aliases` to map nouns like "function requirement" and "test cases" to metamodel item types, then picks a named `relation_whitelist` (or a union) based on target types + intent keywords. The root is pulled from the question text (node id if present, otherwise quoted name with optional type hint). Chosen because it costs zero tokens and never drifts across runs.

**Pure schema (not question-specific)**
- `data/metamodel.json` holds only vocabulary, relation schema (from/to types per sid), named relation whitelists that mirror SystemWeaver's `SidsToFollow` (requirement_trace, test_trace, structural_containment, all_trace), aliases, intent keywords, and generic guard rails (`max_nodes=150`, `max_edges=300` — reach is bounded by the chosen whitelist, not a hop count). No per-question roots, targets, or relation lists.

**Data-driven **
- Extraction, packet assembly, LLM call, grounding check, and artifact I/O.

### Possible Production replacement for the planner

Swap `plan_question()` with ONE of:
1. A small LLM planner that receives ONLY the metamodel and the question text (never node contents) and emits a `QuestionPolicy` JSON. The analysis LLM still sees only the bounded subgraph — the planner LLM never sees the graph, so the bounded-subgraph principle holds.
2. An NER(Named Entity Recognition) + entity-linking pipeline tuned to the organization's naming conventions, abbreviations, and project slang. Recommended when auditability of the planning step matters.

The rule-based planner is limited and used to demo the concept of subgraph extraction. 

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Markdown, display

# sw_trace is installed as an editable package — see pyproject.toml at the
# project root. Run `pip install -e .` from the project root once.
from sw_trace import (
    TraceGraph, Metamodel,
    resolve_llm_config, build_connection_smoke_test_packet, analyze_with_llm,
    run_from_question,
    UsageTracker, format_usage,
    RunBundle, rebuild_token_summary,
    GRAPH_PATH, METAMODEL_PATH, GROUND_TRUTH_PATH,
)

In [ ]:
# --- Config ---
LLM_PROVIDER = "openai"         # "openai" or "lmstudio"
LLM_MODEL = "gpt-5.1"            # e.g. "gpt-5.1", "gpt-5-mini", or "qwen3-14b@q3_k_l"
LMSTUDIO_BASE_URL = "http://localhost:1234/v1"

TEMPERATURE = 0.0
MAX_TOKENS = 4000

# --- Output layout ---
# "per_question" : each question writes its own logs/q<N>_*.json + output/q<N>_answer.md.
#                  Use this when walking through questions one at a time in a workshop.
# "aggregated"   : per-question files are NOT written; instead one
#                  logs/run_<ts>_<mode>.json + output/run_<ts>_<mode>.md bundles all
#                  questions. Use this when running end-to-end in one pass.
# In both layouts, logs/token_usage_ledger.jsonl appends one row per question and
# logs/token_usage_summary.json is refreshed at the end.
RUN_LAYOUT = "per_question"
MODE_LABEL = f"{LLM_PROVIDER}_{LLM_MODEL}".replace(".", "").replace("-", "")

# GRAPH_PATH / METAMODEL_PATH / GROUND_TRUTH_PATH are imported from sw_trace.paths;
# they resolve to the project-rooted data/ and eval/ directories regardless of
# where this notebook is launched from.
for p in [GRAPH_PATH, METAMODEL_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

graph = TraceGraph.from_json(GRAPH_PATH)
metamodel = Metamodel.load(METAMODEL_PATH)

display(Markdown(
    f"**Loaded graph:** {len(graph.nodes):,} nodes, {len(graph.edges):,} edges  \n"
    f"**Metamodel:** {len(metamodel.vocabulary_item_types)} item types, "
    f"{len(metamodel.vocabulary_relation_sids)} relation sids, "
    f"{len(metamodel.type_aliases)} alias groups, "
    f"{len(metamodel.relation_whitelists)} named whitelists  \n"
    f"**Run layout:** `{RUN_LAYOUT}`  \n"
    f"**Mode label:** `{MODE_LABEL}`  \n"
    f"**Planner:** rule-based (workshop shortcut). See `plan_question()` in sw_trace.planner."
))

In [ ]:
# Bundle gathers every question's run. Writes are driven by RUN_LAYOUT set above:
#   "per_question" -> bundle.add() writes per-question files immediately
#   "aggregated"   -> bundle.finalize() writes one run-level JSON + MD at the end
# In both layouts, the token ledger appends one row per question.
bundle = RunBundle(
    mode=MODE_LABEL,
    graph_path=str(GRAPH_PATH),
    metamodel_path=str(METAMODEL_PATH),
    layout=RUN_LAYOUT,
)
usage_tracker = UsageTracker()

display(Markdown(f"**Run id:** `{bundle.run_id}`"))

def render_run(run):
    """Render one question run: planner, compact view, warnings, LLM answer, grounding, tokens."""
    display(Markdown(f"### {run['run_label']}"))
    display(Markdown(f"**Question:** {run['question_text']}"))

    display(Markdown("**Planner inferred (rule-based)**"))
    print(json.dumps(run["planner_diagnostics"], indent=2, ensure_ascii=False))

    display(Markdown("**Compact view (bounded subgraph)**"))
    print(json.dumps(run["compact_view"], indent=2, ensure_ascii=False))

    warnings = run["evidence_packet"]["candidate_subgraph"].get("warnings") or []
    if warnings:
        display(Markdown("**Extraction warnings**"))
        print(json.dumps(warnings, indent=2, ensure_ascii=False))
    else:
        display(Markdown("_No extraction warnings._"))

    display(Markdown(f"**LLM status:** {run['llm_status']}"))

    if run["llm_answer"] is not None:
        display(Markdown("**LLM answer (parsed JSON)**"))
        print(json.dumps(run["llm_answer"], indent=2, ensure_ascii=False))
    elif run["llm_response"] is not None:
        display(Markdown("**LLM raw text (JSON parse failed)**"))
        print(run["llm_response"].get("content", ""))

    if run["llm_validation"] is not None:
        v = run["llm_validation"]
        flag = "OK" if v.get("grounded") else "REVIEW REQUIRED"
        display(Markdown(f"**Grounding: {flag}**"))
        print(json.dumps(v, indent=2, ensure_ascii=False))

    this_usage = usage_tracker.record(run)
    display(Markdown(
        f"**Tokens (this call):** {format_usage(this_usage)}  \n"
        f"**Tokens (running total):** {format_usage(usage_tracker.totals)}"
    ))

    # bundle.add() writes per-question files (layout=per_question) OR just
    # stores the run until finalize() (layout=aggregated). It also appends
    # to the token ledger in both cases.
    bundle.add(run)

def ask(question_text, run_label):
    run = run_from_question(
        graph, metamodel, question_text,
        llm_provider=LLM_PROVIDER,
        llm_model=LLM_MODEL,
        llm_base_url=LMSTUDIO_BASE_URL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        run_label=run_label,
    )
    render_run(run)
    return run

## Tiny LLM connection smoke test

In [ ]:
cfg = resolve_llm_config(LLM_PROVIDER, LLM_MODEL, lmstudio_base_url=LMSTUDIO_BASE_URL)
smoke_packet = build_connection_smoke_test_packet(graph)
print(json.dumps(smoke_packet, indent=2, ensure_ascii=False))
try:
    smoke_response = analyze_with_llm(
        provider=cfg['provider'],
        model=cfg['model'],
        packet=smoke_packet,
        api_key=cfg.get('api_key',''),
        base_url=cfg.get('base_url',''),
        temperature=0.0,
        max_tokens=200,
    )
    display(Markdown("**Smoke-test LLM response**"))
    print(smoke_response['content'])
except Exception as exc:
    display(Markdown(f"**Smoke-test status:** {exc}"))

## Q1

In [ ]:
run_q1 = ask(
    'Which function and design requirements will get affected if the legal requirement "UNECE Regulation No.155" gets removed?',
    run_label="Q1",
)

## Q2

In [ ]:
run_q2 = ask(
    'Does the requirement "Keyless entry" (x0400000000038EAE) have any test cases?',
    run_label="Q2",
)

## Q3

In [ ]:
run_q3 = ask(
    'Starting from the Stakeholder Requirement "Unauthorized start detection", which Function Requirements appear in the local downstream trace tree, and which Design Requirements appear one level further downstream?',
    run_label="Q3",
)

## Q4

In [ ]:
run_q4 = ask(
    'If the Function Requirement "Engine start time" is tightened (smaller N), which Design Requirements are impacted downstream? Optionally: Which of them mention timing or timeout explicitly?',
    run_label="Q4",
)

## Token usage summary

In [ ]:
print(usage_tracker.summary())
print()
written = bundle.finalize(ground_truth_path=GROUND_TRUTH_PATH)
for k, v in written.items():
    print(f"{k}: {v}")